In [1]:
import json
import math
import itertools
from dataclasses import dataclass
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

# tqdm is optional but highly recommended for multi-seed progress reporting
try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("tqdm not found — install with: pip install tqdm  (progress bars disabled)")

In [2]:
# =========================
# PATHS (EDIT HERE)
# =========================
DATA_DIR = Path(r"C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned")  # <--
CANONICAL_FILE = "pak_events_canonical.parquet"  # <--

# Strategy name determines exclusive output directories
STRATEGY_NAME = "cumulative"  # <--

# Root output folder (exclusive per strategy)
OUT_ROOT = DATA_DIR / "_grid_outputs" / STRATEGY_NAME  # <--
BOUNDED_DIR = OUT_ROOT / "bounded"
DP_DIR = OUT_ROOT / "dp_outputs"
CSV_DIR = OUT_ROOT / "metrics_csv"
PLOTS_DIR = OUT_ROOT / "plots"

for d in [BOUNDED_DIR, DP_DIR, CSV_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =========================
# SLICE GRID (EDIT HERE)
# =========================
WINDOW_DAYS_LIST = [15, 30, 45, 60, 90, 120, 180]  # <--
for x in WINDOW_DAYS_LIST:
    SLIDE_STEP_DAYS = x // 2  # <-- (sliding step)
    TOPK = min(10, x // 3)  # <-- TOP-K on daily series (top-k days)

# Optional cap (avoid too many slices while testing)
MAX_SLICES_PER_WINDOW = None  # e.g., 10 or None  # <--

# =========================
# BOUNDING GRID (EDIT HERE)
# =========================
MAX_ORDERS_PER_DAY_LIST = [1, 2, 3]  # <--
REV_CLIP_PCT_LIST = [70, 80, 90, 95]  # <-- percentile for per-user-per-day revenue cap B

# =========================
# DP GRID (EDIT HERE)
# =========================
EPS_TOTALS = [0.25, 0.5, 1, 2, 4, 8, 16]  # <--

# =========================
# MULTI-TRIAL SEED CONFIG  ← PHASE 1 ADDITION
# =========================
BASE_SEED = 42    # <-- seed for seed_id=0; each trial uses BASE_SEED + seed_id
NUM_SEEDS = 50    # <-- number of independent noise trials (30-50 recommended)

# =========================
# DECISION RULE PARAMS (EDIT HERE)
# =========================
THRESH_QUANTILE = 0.75  # <-- threshold alarm uses truth quantile per slice/metric
TREND_WINDOW_DAYS = 7   # <--
CP_DIFF_MULTIPLIER = 2.0  # <-- threshold = multiplier * std(diff(truth))
CP_TOLERANCE_DAYS = 3   # <-- matching tolerance for change-point F1 (±days)


In [3]:
canonical_path = DATA_DIR / CANONICAL_FILE
assert canonical_path.exists(), f"Canonical file not found: {canonical_path}"

events = pd.read_parquet(canonical_path)

# Required columns assumed: ts, date, user_id, value, order_id (or transaction id)
# If your canonical has different names, edit here:
COL_TS = "ts"         # <--
COL_DATE = "date"     # <--
COL_USER = "user_id"  # <--
COL_VALUE = "value"   # <--
COL_ORDER = "order_id"  # <--

events[COL_TS] = pd.to_datetime(events[COL_TS], errors="coerce")
events[COL_DATE] = pd.to_datetime(events[COL_DATE], errors="coerce").dt.floor("D")
events[COL_USER] = events[COL_USER].astype("string")
events[COL_VALUE] = pd.to_numeric(events[COL_VALUE], errors="coerce").fillna(0.0)

# order_id might be missing for non-order events; keep as is.
if COL_ORDER in events.columns:
    events[COL_ORDER] = events[COL_ORDER].astype("string")

min_date = events[COL_DATE].min()
max_date = events[COL_DATE].max()
assert pd.notna(min_date) and pd.notna(max_date), "Dates missing after parsing."

print("Loaded:", events.shape)
print("Date range:", min_date.date(), "to", max_date.date())
print("Columns:", list(events.columns))

Loaded: (169787, 6)
Date range: 2016-07-01 to 2018-08-28
Columns: ['ts', 'date', 'user_id', 'event_type', 'value', 'order_id']


In [4]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def pearson_corr(a: pd.Series, b: pd.Series) -> float:
    a = a.astype(float)
    b = b.astype(float)
    if a.std() == 0 or b.std() == 0:
        return float("nan")
    return float(a.corr(b))

def make_daily_bounded_truth(
    ev: pd.DataFrame,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    max_orders_per_user_day: int,
    rev_clip_pct: float,
) -> tuple[pd.DataFrame, float]:
    """
    Returns:
      daily_truth: index=date, cols=[ORDERS, REVENUE, DAU]
      B: per-user-per-day revenue clipping cap (used as sensitivity for REVENUE)
    """
    sl = ev[(ev[COL_DATE] >= start_date) & (ev[COL_DATE] <= end_date)].copy()

    # DAU: per-user-day presence (cap 1)
    user_day_presence = (
        sl.groupby([COL_USER, COL_DATE])
          .size()
          .clip(upper=1)
          .reset_index(name="dau_contrib")
    )

    # ORDERS: per-user-day unique orders, clipped to k
    if COL_ORDER in sl.columns:
        user_day_orders = (
            sl.groupby([COL_USER, COL_DATE])[COL_ORDER]
              .nunique(dropna=True)
              .fillna(0)
              .clip(upper=max_orders_per_user_day)
              .reset_index(name="orders_contrib")
        )
    else:
        user_day_orders = (
            sl.groupby([COL_USER, COL_DATE])
              .size()
              .clip(upper=max_orders_per_user_day)
              .reset_index(name="orders_contrib")
        )

    # REVENUE: per-user-day sum(value), clipped to B
    user_day_rev_raw = sl.groupby([COL_USER, COL_DATE])[COL_VALUE].sum()
    B = float(user_day_rev_raw.quantile(rev_clip_pct / 100.0)) if len(user_day_rev_raw) else 0.0
    user_day_revenue = (
        user_day_rev_raw.clip(upper=B)
                        .reset_index(name="revenue_contrib")
    )

    merged = user_day_presence.merge(user_day_orders, on=[COL_USER, COL_DATE], how="outer") \
                              .merge(user_day_revenue, on=[COL_USER, COL_DATE], how="outer") \
                              .fillna(0)

    daily = merged.groupby(COL_DATE).agg(
        DAU=("dau_contrib", "sum"),
        ORDERS=("orders_contrib", "sum"),
        REVENUE=("revenue_contrib", "sum"),
    )

    full_idx = pd.date_range(start_date, end_date, freq="D")
    daily = daily.reindex(full_idx).fillna(0)
    daily.index.name = "date"

    return daily, B

# -------------------------
# DP: Naive per-release Laplace
# -------------------------
def dp_naive_laplace(daily_truth: pd.DataFrame, eps_total: float, sens: dict[str, float], rng: np.random.Generator) -> pd.DataFrame:
    T = len(daily_truth)
    eps_t = eps_total / T
    out = pd.DataFrame(index=daily_truth.index)

    for metric in ["DAU", "ORDERS", "REVENUE"]:
        scale = sens[metric] / eps_t if eps_t > 0 else float("inf")
        noise = rng.laplace(loc=0.0, scale=scale, size=T)
        out[f"{metric}_DP"] = (daily_truth[metric].to_numpy(dtype=float) + noise)

    return out.clip(lower=0)

# -------------------------
# DP: Binary Tree (Fenwick) for prefix sums, then diff to daily
# -------------------------
def fenwick_build(arr: np.ndarray) -> np.ndarray:
    n = len(arr)
    bit = np.zeros(n + 1, dtype=float)
    for i in range(1, n + 1):
        j = i
        while j <= n:
            bit[j] += arr[i - 1]
            j += j & -j
    return bit

def fenwick_prefix(bit: np.ndarray, i: int) -> float:
    s = 0.0
    while i > 0:
        s += bit[i]
        i -= i & -i
    return s

def dp_tree_prefix_laplace(values: np.ndarray, eps_total: float, sensitivity: float, rng: np.random.Generator) -> np.ndarray:
    T = len(values)
    if T == 0:
        return np.array([], dtype=float)

    levels = int(math.floor(math.log2(T))) + 1
    eps_per_level = eps_total / levels

    bit = fenwick_build(values)
    noisy_bit = bit.copy()
    for idx in range(1, T + 1):
        noisy_bit[idx] = bit[idx] + rng.laplace(0.0, sensitivity / eps_per_level)

    noisy_prefix = np.zeros(T, dtype=float)
    for t in range(1, T + 1):
        noisy_prefix[t - 1] = fenwick_prefix(noisy_bit, t)

    return noisy_prefix

def dp_binary_tree(daily_truth: pd.DataFrame, eps_total: float, sens: dict[str, float], rng: np.random.Generator) -> pd.DataFrame:
    out = pd.DataFrame(index=daily_truth.index)
    for metric in ["DAU", "ORDERS", "REVENUE"]:
        pref = dp_tree_prefix_laplace(daily_truth[metric].to_numpy(dtype=float), eps_total, sens[metric], rng)
        daily = np.diff(np.concatenate([[0.0], pref]))
        out[f"{metric}_TREE_DP"] = daily
    return out.clip(lower=0)

# -------------------------
# DP: Smooth Binary (bitwise noise reuse) -> prefix, then diff
# -------------------------
def dp_smooth_prefix(values: np.ndarray, eps_total: float, sensitivity: float, rng: np.random.Generator) -> np.ndarray:
    T = len(values)
    if T == 0:
        return np.array([], dtype=float)

    L = int(math.ceil(math.log2(T))) if T > 1 else 1
    eps_per_level = eps_total / L

    noise_for_bit = np.array([rng.laplace(0.0, sensitivity / eps_per_level) for _ in range(L)], dtype=float)

    noisy_prefix = np.zeros(T, dtype=float)
    running = 0.0
    for i in range(T):
        running += values[i]
        s = running
        idx = i + 1
        bit = 0
        while idx > 0 and bit < L:
            if idx & 1:
                s += noise_for_bit[bit]
            idx >>= 1
            bit += 1
        noisy_prefix[i] = s
    return noisy_prefix

def dp_smooth_binary(daily_truth: pd.DataFrame, eps_total: float, sens: dict[str, float], rng: np.random.Generator) -> pd.DataFrame:
    out = pd.DataFrame(index=daily_truth.index)
    for metric in ["DAU", "ORDERS", "REVENUE"]:
        pref = dp_smooth_prefix(daily_truth[metric].to_numpy(dtype=float), eps_total, sens[metric], rng)
        daily = np.diff(np.concatenate([[0.0], pref]))
        out[f"{metric}_SMOOTH_DP"] = daily
    return out.clip(lower=0)

In [5]:
def decision_stats(truth_labels: pd.Series, dp_labels: pd.Series) -> dict:
    truth = truth_labels.to_numpy(dtype=int)
    pred = dp_labels.to_numpy(dtype=int)
    assert truth.shape == pred.shape

    tn = int(np.sum((truth == 0) & (pred == 0)))
    fp = int(np.sum((truth == 0) & (pred == 1)))
    fn = int(np.sum((truth == 1) & (pred == 0)))
    tp = int(np.sum((truth == 1) & (pred == 1)))
    total = len(truth)

    return {
        "flip_rate": (fp + fn) / total if total else float("nan"),
        "false_positive_rate": fp / total if total else float("nan"),
        "false_negative_rate": fn / total if total else float("nan"),
        "TP": tp, "FP": fp, "TN": tn, "FN": fn
    }

def threshold_alarm(series: pd.Series, threshold: float) -> pd.Series:
    return (series > threshold).astype(int)

def trend_positive(series: pd.Series, window_days: int = 7) -> pd.Series:
    labels = pd.Series(0, index=series.index)
    diffs = series.diff()
    for t in range(window_days, len(series)):
        window = diffs.iloc[t - window_days + 1: t + 1].dropna()
        if len(window) == window_days and (window > 0).all():
            labels.iloc[t] = 1
    return labels

def change_points_by_diff(series: pd.Series, threshold: float) -> pd.Series:
    d = series.diff().abs().fillna(0)
    return (d > threshold).astype(int)

def cp_f1_with_tolerance(truth_cp: pd.Series, pred_cp: pd.Series, tol_days: int) -> tuple[float, float]:
    truth_idx = np.where(truth_cp.to_numpy(dtype=int) == 1)[0]
    pred_idx = np.where(pred_cp.to_numpy(dtype=int) == 1)[0]

    if len(truth_idx) == 0 and len(pred_idx) == 0:
        return (1.0, 0.0)
    if len(truth_idx) == 0:
        return (0.0, float("nan"))
    if len(pred_idx) == 0:
        return (0.0, float("nan"))

    used_truth = set()
    tp = 0
    delays = []
    for p in pred_idx:
        candidates = [t for t in truth_idx if t not in used_truth and abs(t - p) <= tol_days]
        if candidates:
            t_best = min(candidates, key=lambda t: abs(t - p))
            used_truth.add(t_best)
            tp += 1
            delays.append(p - t_best)

    fp = len(pred_idx) - tp
    fn = len(truth_idx) - tp

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    avg_delay = float(np.mean(delays)) if delays else float("nan")
    return (f1, avg_delay)

def overlap_at_k(truth_series: pd.Series, dp_series: pd.Series, k: int) -> float:
    k = min(k, len(truth_series))
    if k == 0:
        return float("nan")
    top_truth = set(truth_series.nlargest(k).index)
    top_dp = set(dp_series.nlargest(k).index)
    return len(top_truth & top_dp) / k

def kendall_tau_on_ranks(truth_series: pd.Series, dp_series: pd.Series) -> float:
    a = truth_series.rank(method="average", ascending=False).to_numpy()
    b = dp_series.rank(method="average", ascending=False).to_numpy()
    n = len(a)
    if n < 2:
        return float("nan")

    concordant = 0
    discordant = 0
    for i in range(n):
        for j in range(i + 1, n):
            s1 = np.sign(a[i] - a[j])
            s2 = np.sign(b[i] - b[j])
            if s1 == 0 or s2 == 0:
                continue
            if s1 == s2:
                concordant += 1
            else:
                discordant += 1
    denom = concordant + discordant
    return (concordant - discordant) / denom if denom else float("nan")

def eval_pointwise_and_trend(truth: pd.Series, dp: pd.Series) -> dict:
    mae = float(mean_absolute_error(truth, dp))
    rm = rmse(truth, dp)
    corr_level = pearson_corr(truth, dp)
    corr_diff = pearson_corr(truth.diff().fillna(0), dp.diff().fillna(0))
    return {"MAE": mae, "RMSE": rm, "CORR_LEVEL": corr_level, "CORR_DIFF": corr_diff}

In [6]:
def gen_slices_cumulative(min_date: pd.Timestamp, max_date: pd.Timestamp, horizons_days: list[int]):
    slices = []
    for h in horizons_days:
        end = min_date + timedelta(days=h - 1)
        if end <= max_date:
            slices.append((min_date, end))
    return slices

slice_jobs = []
for (s, e) in gen_slices_cumulative(min_date, max_date, WINDOW_DAYS_LIST):
    wdays = (e - s).days + 1
    slice_jobs.append({"window_days": wdays, "start": s, "end": e})

print("Total cumulative slice jobs:", len(slice_jobs))
print("All:", slice_jobs)
print(f"\nTotal rows to compute: {len(slice_jobs)} slices × {len(MAX_ORDERS_PER_DAY_LIST)} max_k × {len(REV_CLIP_PCT_LIST)} rev_pct × {len(EPS_TOTALS)} eps × 3 mechs × 3 metrics × {NUM_SEEDS} seeds")
print(f"  ≈ {len(slice_jobs) * len(MAX_ORDERS_PER_DAY_LIST) * len(REV_CLIP_PCT_LIST) * len(EPS_TOTALS) * 3 * 3 * NUM_SEEDS:,} rows")

Total cumulative slice jobs: 7
All: [{'window_days': 15, 'start': Timestamp('2016-07-01 00:00:00'), 'end': Timestamp('2016-07-15 00:00:00')}, {'window_days': 30, 'start': Timestamp('2016-07-01 00:00:00'), 'end': Timestamp('2016-07-30 00:00:00')}, {'window_days': 45, 'start': Timestamp('2016-07-01 00:00:00'), 'end': Timestamp('2016-08-14 00:00:00')}, {'window_days': 60, 'start': Timestamp('2016-07-01 00:00:00'), 'end': Timestamp('2016-08-29 00:00:00')}, {'window_days': 90, 'start': Timestamp('2016-07-01 00:00:00'), 'end': Timestamp('2016-09-28 00:00:00')}, {'window_days': 120, 'start': Timestamp('2016-07-01 00:00:00'), 'end': Timestamp('2016-10-28 00:00:00')}, {'window_days': 180, 'start': Timestamp('2016-07-01 00:00:00'), 'end': Timestamp('2016-12-27 00:00:00')}]

Total rows to compute: 7 slices × 3 max_k × 4 rev_pct × 7 eps × 3 mechs × 3 metrics × 50 seeds
  ≈ 264,600 rows


In [7]:
def save_json(path: Path, obj: dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)

rows = []

# ── OUTER SEED LOOP ─────────────────────────────────────────────────────────
# Each seed_id draws a fresh, independent RNG → independent noise realisation.
# Bounded truth files and DP parquet artefacts are written only for seed_id==0
# to avoid 50× disk bloat; the metrics CSV captures all trials.
# ─────────────────────────────────────────────────────────────────────────────
seed_iter = range(NUM_SEEDS)
if HAS_TQDM:
    seed_iter = tqdm(seed_iter, desc="Seeds", unit="seed")

for seed_id in seed_iter:
    rng = np.random.default_rng(BASE_SEED + seed_id)  # independent stream per trial

    for sj in slice_jobs:
        s = sj["start"]
        e = sj["end"]
        wdays = sj["window_days"]

        for max_k, rev_pct in itertools.product(MAX_ORDERS_PER_DAY_LIST, REV_CLIP_PCT_LIST):
            daily_truth, B = make_daily_bounded_truth(events, s, e, max_k, rev_pct)

            sens = {
                "DAU": 1.0,
                "ORDERS": float(max_k),
                "REVENUE": float(B),
            }

            bounded_tag = f"s={s.date()}__e={e.date()}__win={wdays}__k={max_k}__revpct={rev_pct}"

            # Save bounded truth artefacts only once (deterministic, seed-independent)
            if seed_id == 0:
                bounded_path = BOUNDED_DIR / f"bounded__{bounded_tag}.parquet"
                daily_truth.to_parquet(bounded_path)
                save_json(bounded_path.with_suffix(".json"), {
                    "strategy": STRATEGY_NAME,
                    "slice_start": str(s.date()),
                    "slice_end": str(e.date()),
                    "window_days": wdays,
                    "max_orders_per_user_day": max_k,
                    "rev_clip_pct": rev_pct,
                    "B_revenue_cap": B,
                    "sensitivities": sens,
                })

            thresh = {
                "DAU": float(daily_truth["DAU"].quantile(THRESH_QUANTILE)),
                "ORDERS": float(daily_truth["ORDERS"].quantile(THRESH_QUANTILE)),
                "REVENUE": float(daily_truth["REVENUE"].quantile(THRESH_QUANTILE)),
            }

            cp_thresh = {}
            for metric in ["DAU", "ORDERS", "REVENUE"]:
                sd = float(daily_truth[metric].diff().fillna(0).std())
                cp_thresh[metric] = CP_DIFF_MULTIPLIER * sd

            for eps in EPS_TOTALS:
                dp_naive = dp_naive_laplace(daily_truth, eps, sens, rng)
                dp_tree = dp_binary_tree(daily_truth, eps, sens, rng)
                dp_smooth = dp_smooth_binary(daily_truth, eps, sens, rng)

                dp_pack = {
                    "naive": dp_naive,
                    "tree": dp_tree,
                    "smooth": dp_smooth,
                }

                # Save DP parquet artefacts only for seed_id==0
                if seed_id == 0:
                    for mech, dpdf in dp_pack.items():
                        dp_path = DP_DIR / f"dp__{mech}__eps={eps}__{bounded_tag}.parquet"
                        dpdf.to_parquet(dp_path)
                        save_json(dp_path.with_suffix(".json"), {
                            "strategy": STRATEGY_NAME,
                            "mechanism": mech,
                            "eps_total": eps,
                            "bounded_ref": f"bounded__{bounded_tag}.parquet",
                            "slice_start": str(s.date()),
                            "slice_end": str(e.date()),
                            "window_days": wdays,
                            "max_orders_per_user_day": max_k,
                            "rev_clip_pct": rev_pct,
                            "B_revenue_cap": B,
                            "seed_id": seed_id,
                        })

                for mech, dpdf in dp_pack.items():
                    colmap = {
                        "naive":  {"DAU": "DAU_DP", "ORDERS": "ORDERS_DP", "REVENUE": "REVENUE_DP"},
                        "tree":   {"DAU": "DAU_TREE_DP", "ORDERS": "ORDERS_TREE_DP", "REVENUE": "REVENUE_TREE_DP"},
                        "smooth": {"DAU": "DAU_SMOOTH_DP", "ORDERS": "ORDERS_SMOOTH_DP", "REVENUE": "REVENUE_SMOOTH_DP"},
                    }[mech]

                    for metric in ["DAU", "ORDERS", "REVENUE"]:
                        truth_series = daily_truth[metric].astype(float)
                        dp_series = dpdf[colmap[metric]].astype(float)

                        pt = eval_pointwise_and_trend(truth_series, dp_series)

                        truth_alarm = threshold_alarm(truth_series, thresh[metric])
                        dp_alarm = threshold_alarm(dp_series, thresh[metric])
                        alarm_stats = decision_stats(truth_alarm, dp_alarm)

                        truth_trend = trend_positive(truth_series, TREND_WINDOW_DAYS)
                        dp_trend = trend_positive(dp_series, TREND_WINDOW_DAYS)
                        trend_stats = decision_stats(truth_trend, dp_trend)

                        truth_cp = change_points_by_diff(truth_series, cp_thresh[metric])
                        dp_cp = change_points_by_diff(dp_series, cp_thresh[metric])
                        cp_f1, cp_delay = cp_f1_with_tolerance(truth_cp, dp_cp, CP_TOLERANCE_DAYS)

                        ovk = overlap_at_k(truth_series, dp_series, TOPK)
                        tau = kendall_tau_on_ranks(truth_series, dp_series)

                        rows.append({
                            # ── trial identifier ────────────────────────────
                            "seed": seed_id,  # ← NEW: independent noise trial
                            # ── experimental config ─────────────────────────
                            "strategy": STRATEGY_NAME,
                            "slice_start": str(s.date()),
                            "slice_end": str(e.date()),
                            "window_days": wdays,
                            "max_orders_per_user_day": max_k,
                            "rev_clip_pct": rev_pct,
                            "B_revenue_cap": B,
                            "eps_total": eps,
                            "mechanism": mech,
                            "metric": metric,
                            # ── A) pointwise ────────────────────────────────
                            "MAE": pt["MAE"],
                            "RMSE": pt["RMSE"],
                            # ── B) trend/shape ──────────────────────────────
                            "CORR_LEVEL": pt["CORR_LEVEL"],
                            "CORR_DIFF": pt["CORR_DIFF"],
                            # ── C) threshold alarm ──────────────────────────
                            "THRESH": thresh[metric],
                            "ALARM_flip": alarm_stats["flip_rate"],
                            "ALARM_fp_rate": alarm_stats["false_positive_rate"],
                            "ALARM_fn_rate": alarm_stats["false_negative_rate"],
                            # ── C) trend decision ───────────────────────────
                            "TREND_flip": trend_stats["flip_rate"],
                            "TREND_fp_rate": trend_stats["false_positive_rate"],
                            "TREND_fn_rate": trend_stats["false_negative_rate"],
                            # ── C) change-point ─────────────────────────────
                            "CP_thresh": cp_thresh[metric],
                            "CP_f1": cp_f1,
                            "CP_avg_delay": cp_delay,
                            # ── C) top-k stability on days ──────────────────
                            "TOPK": TOPK,
                            "OVERLAP_at_k": ovk,
                            "KENDALL_tau": tau,
                        })

print(f"Done. Total metric rows: {len(rows):,}  ({NUM_SEEDS} seeds × {len(rows)//NUM_SEEDS:,} rows/seed)")

Seeds: 100%|██████████| 50/50 [2:27:57<00:00, 177.54s/seed]  

Done. Total metric rows: 264,600  (50 seeds × 5,292 rows/seed)


In [8]:
results_df = pd.DataFrame(rows)

# ── Save raw multi-trial CSV ─────────────────────────────────────────────────
csv_path = CSV_DIR / "results_cumulative_multi.csv"
results_df.to_csv(csv_path, index=False)
print("Saved raw multi-trial CSV:", csv_path)
print(f"Shape: {results_df.shape}")

# ── Quick sanity: per-seed row counts should all be equal ────────────────────
seed_counts = results_df.groupby("seed").size()
assert seed_counts.nunique() == 1, "Unequal rows per seed — check for missing combos!"
print(f"Rows per seed: {seed_counts.iloc[0]:,}  ✓")

results_df.head()

Saved raw multi-trial CSV: C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\metrics_csv\results_cumulative_multi.csv
Shape: (264600, 28)
Rows per seed: 5,292  ✓


,seed,strategy,slice_start,slice_end,window_days,max_orders_per_user_day,rev_clip_pct,B_revenue_cap,eps_total,mechanism,...,ALARM_fn_rate,TREND_flip,TREND_fp_rate,TREND_fn_rate,CP_thresh,CP_f1,CP_avg_delay,TOPK,OVERLAP_at_k,KENDALL_tau
0,0,cumulative,2016-07-01,2016-07-15,15,1,70,1624.5,0.25,naive,...,0.066667,0.0,0.0,0.0,45.522156,0.200000,-2.0,10,0.8,0.242718
1,0,cumulative,2016-07-01,2016-07-15,15,1,70,1624.5,0.25,naive,...,0.200000,0.0,0.0,0.0,45.522156,0.181818,-3.0,10,0.6,-0.009901
2,0,cumulative,2016-07-01,2016-07-15,15,1,70,1624.5,0.25,naive,...,0.000000,0.0,0.0,0.0,48705.709580,0.285714,-1.0,10,0.9,0.509804
3,0,cumulative,2016-07-01,2016-07-15,15,1,70,1624.5,0.25,tree,...,0.133333,0.0,0.0,0.0,45.522156,0.250000,-3.0,10,0.9,0.538462
4,0,cumulative,2016-07-01,2016-07-15,15,1,70,1624.5,0.25,tree,...,0.133333,0.0,0.0,0.0,45.522156,0.500000,-1.0,10,0.9,0.480769


In [9]:
# ── Mean ± std plots across seeds ────────────────────────────────────────────

MECH_COLORS = {"naive": "#e15759", "tree": "#4e79a7", "smooth": "#59a14f"}

def save_mean_std_plot(
    df: pd.DataFrame,
    x: str,
    y: str,
    hue: str,
    title: str,
    filename: str,
    y_label: str = None,
):
    """Line plot with ±1 std shaded band; each curve is a mechanism."""
    agg = (
        df.groupby([x, hue])[y]
          .agg(["mean", "std"])
          .reset_index()
    )
    agg["std"] = agg["std"].fillna(0.0)

    fig, ax = plt.subplots(figsize=(10, 5))
    for key, g in agg.groupby(hue):
        g = g.sort_values(x)
        color = MECH_COLORS.get(str(key), None)
        ax.plot(g[x], g["mean"], marker="o", label=str(key), color=color)
        ax.fill_between(
            g[x],
            g["mean"] - g["std"],
            g["mean"] + g["std"],
            alpha=0.18,
            color=color,
        )

    ax.set_title(f"{title}\n(mean ± 1 std, N={NUM_SEEDS} seeds)")
    ax.set_xlabel(x)
    ax.set_ylabel(y_label or y)
    ax.legend(title=hue)
    out = PLOTS_DIR / filename
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)
    print("Saved plot:", out)


tmp = results_df[results_df["metric"] == "ORDERS"]
save_mean_std_plot(
    tmp, x="eps_total", y="MAE", hue="mechanism",
    title="ORDERS: Mean MAE vs ε_total (avg over horizons)",
    filename="orders_mae_vs_eps_multi.png",
)
save_mean_std_plot(
    tmp, x="eps_total", y="ALARM_flip", hue="mechanism",
    title="ORDERS Threshold Alarm: Flip rate vs ε_total",
    filename="orders_alarm_flip_vs_eps_multi.png",
)
save_mean_std_plot(
    tmp, x="eps_total", y="TREND_flip", hue="mechanism",
    title="ORDERS Trend(7d): Flip rate vs ε_total",
    filename="orders_trend_flip_vs_eps_multi.png",
)
save_mean_std_plot(
    tmp, x="eps_total", y="CP_f1", hue="mechanism",
    title="ORDERS Change-point: F1 vs ε_total",
    filename="orders_cp_f1_vs_eps_multi.png",
)

tmp_rev = results_df[results_df["metric"] == "REVENUE"]
save_mean_std_plot(
    tmp_rev, x="eps_total", y="MAE", hue="mechanism",
    title="REVENUE: Mean MAE vs ε_total (avg over horizons)",
    filename="revenue_mae_vs_eps_multi.png",
)

print("All plots saved.")

Saved plot: C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\orders_mae_vs_eps_multi.png
Saved plot: C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\orders_alarm_flip_vs_eps_multi.png
Saved plot: C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\orders_trend_flip_vs_eps_multi.png
Saved plot: C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\orders_cp_f1_vs_eps_multi.png
Saved plot: C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\revenue_mae_vs_eps_multi.png
All plots saved.


## Phase 1 — Remaining Analyses

The following cells implement the four remaining Phase 1 tasks. **None of them re-run DP computations** — they all operate on already-produced outputs:

1. **RDP Composition Accounting** — analytical derivation from grid parameters (T, ε, mechanism)
2. **Clipping Bias Analysis** — reads seed_id=0 bounded + DP parquets already saved to disk
3. **ε vs T Heatmaps (Safe Zone Visualization)** — pivots the existing multi-trial CSV on `window_days`
4. **Privacy–Utility Pareto Frontier** — aggregates existing CSV metrics into a composite utility score

### Task 1 — RDP Composition Accounting

All three mechanisms declare the same `ε_total` privacy budget, but they achieve it through very
different per-query allocations:

- **Naive:** T sequential Laplace queries, each at `ε_t = ε_total / T`
- **Binary Tree:** L = ⌊log₂T⌋ + 1 levels, each level at `ε_level = ε_total / L`
- **Smooth Binary:** L = ⌈log₂T⌉ levels, same per-level allocation

Under sequential composition all three compose to exactly `ε_total`. The RDP analysis here serves
two purposes:
1. **Formally verify** the composed (ε, δ)-DP guarantee for each mechanism and window length T.
2. **Derive the theoretical noise variance** per daily estimate, and compare the predicted
   Naive/Smooth improvement ratio to the empirically observed ratio in our multi-seed results.

This forms the theory–empirics alignment argument required for the research paper.

In [13]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# ── RDP for a single Laplace query ───────────────────────────────────────────
# Rényi divergence at order α between Lap(Δ/ε) and the same distribution
# shifted by Δ (sensitivity = 1 normalised throughout).
# Formula (valid for α > 1, integer or real):
#   RDP(α) = 1/(α-1) * log[ α/(2α-1)*exp((α-1)ε) + (α-1)/(2α-1)*exp(-αε) ]

def rdp_laplace_single(alpha, eps_query):
    if alpha <= 1:
        return eps_query
    a, e = alpha, eps_query
    coeff_x = a / (2 * a - 1)
    coeff_y = (a - 1) / (2 * a - 1)
    decay = math.exp(-(2 * a - 1) * e)   # underflows to 0 safely, never overflows
    return ((a - 1) * e + math.log(coeff_x + coeff_y * decay)) / (a - 1)


def rdp_to_eps_delta(rdp_curve: dict, delta: float) -> float:
    """
    Convert an RDP curve {α: rdp(α)} to an (ε, δ)-DP guarantee.
    Uses the Balle et al. (2020) conversion:
      ε = min_{α>1} [ rdp(α) + log(1/δ) / (α - 1) ]
    """
    best = float("inf")
    for alpha, rho in rdp_curve.items():
        if alpha > 1:
            best = min(best, rho + math.log(1.0 / delta) / (alpha - 1))
    return best


# ── Theoretical noise variance per daily estimate ────────────────────────────
# These formulas give the Laplace noise variance contributed to a single
# daily released value (i.e. after differencing prefix sums where applicable).
# Sensitivity is normalised to 1 throughout; multiply by Δ² to recover units.

def noise_std_naive(T: int, eps_total: float) -> float:
    """
    Naive: one Laplace per day with scale = T/ε.
    Std of daily release = T / ε * sqrt(2).
    """
    return (T / eps_total) * math.sqrt(2)


def noise_std_tree(T: int, eps_total: float) -> float:
    """
    Binary Tree: prefix sum uses L nodes each with scale = L/ε.
    Daily release = Prefix[t] - Prefix[t-1].  In the worst case both prefix
    queries share no tree nodes (L independent Laplace per query, 2L total),
    but in practice ~L/2 nodes cancel.  The standard bound uses L distinct
    nodes → Var = L * 2*(L/ε)² → std = L^(3/2) * sqrt(2) / ε.
    """
    L = math.floor(math.log2(T)) + 1 if T > 1 else 1
    return (L ** 1.5) * math.sqrt(2) / eps_total


def noise_std_smooth(T: int, eps_total: float) -> float:
    """Smooth Binary uses L = ceil(log2(T)) — same formula as Tree."""
    L = math.ceil(math.log2(T)) if T > 1 else 1
    return (L ** 1.5) * math.sqrt(2) / eps_total


# ── Sweep over (T, ε) grid ───────────────────────────────────────────────────
T_VALUES   = [15, 30, 45, 60, 90, 120, 180]
EPS_VALUES = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0]
ALPHAS     = list(range(2, 64)) + [128, 256]   # orders for RDP curve
DELTA      = 1e-5                               # target δ for (ε,δ)-DP conversion

rdp_rows = []
for T in T_VALUES:
    L_tree   = math.floor(math.log2(T)) + 1 if T > 1 else 1
    L_smooth = math.ceil(math.log2(T))       if T > 1 else 1

    for eps_total in EPS_VALUES:
        eps_naive  = eps_total / T
        eps_tree   = eps_total / L_tree
        eps_smooth = eps_total / L_smooth

        # Compose RDP curves: each query contributes rdp_laplace_single
        rdp_naive  = {a: T        * rdp_laplace_single(a, eps_naive)  for a in ALPHAS}
        rdp_tree   = {a: L_tree   * rdp_laplace_single(a, eps_tree)   for a in ALPHAS}
        rdp_smooth = {a: L_smooth * rdp_laplace_single(a, eps_smooth) for a in ALPHAS}

        # Formally verified (ε, δ=1e-5)-DP — should match ε_total for Laplace (pure DP)
        eps_d_naive  = rdp_to_eps_delta(rdp_naive,  DELTA)
        eps_d_tree   = rdp_to_eps_delta(rdp_tree,   DELTA)
        eps_d_smooth = rdp_to_eps_delta(rdp_smooth, DELTA)

        # Theoretical noise std and improvement ratio (Naive/structured)
        std_naive  = noise_std_naive(T, eps_total)
        std_tree   = noise_std_tree(T, eps_total)
        std_smooth = noise_std_smooth(T, eps_total)

        rdp_rows.append({
            "T":          T,
            "L_tree":     L_tree,
            "L_smooth":   L_smooth,
            "eps_total":  eps_total,
            # Per-query allocation
            "eps_per_query_naive":   round(eps_naive,  6),
            "eps_per_level_tree":    round(eps_tree,   6),
            "eps_per_level_smooth":  round(eps_smooth, 6),
            # Composed (ε, δ=1e-5) — formal privacy guarantee
            "composed_eps_delta_naive":  round(eps_d_naive,  4),
            "composed_eps_delta_tree":   round(eps_d_tree,   4),
            "composed_eps_delta_smooth": round(eps_d_smooth, 4),
            # Theoretical noise std (sensitivity-normalised)
            "std_naive":   round(std_naive,  4),
            "std_tree":    round(std_tree,   4),
            "std_smooth":  round(std_smooth, 4),
            # Predicted improvement: how many times quieter are structured mechanisms?
            "predicted_ratio_naive_tree":   round(std_naive / std_tree,   3),
            "predicted_ratio_naive_smooth": round(std_naive / std_smooth, 3),
        })

rdp_df = pd.DataFrame(rdp_rows)

# Save
rdp_csv = CSV_DIR / "rdp_composition_analysis.csv"
rdp_df.to_csv(rdp_csv, index=False)
print(f"Saved RDP analysis → {rdp_csv}")

# ── Sanity print: T=15 (our window) ─────────────────────────────────────────
print("\n=== Composed (ε, δ=1e-5) vs declared ε_total — T=15 ===")
cols = ["eps_total", "composed_eps_delta_naive", "composed_eps_delta_tree",
        "predicted_ratio_naive_tree", "predicted_ratio_naive_smooth"]
print(rdp_df[rdp_df["T"] == 15][cols].to_string(index=False))

print("\n=== Theoretical noise std ratio (Naive/Tree) vs T — ε=1.0 ===")
sub = rdp_df[rdp_df["eps_total"] == 1.0][
    ["T", "L_tree", "std_naive", "std_tree", "std_smooth",
     "predicted_ratio_naive_tree", "predicted_ratio_naive_smooth"]
]
print(sub.to_string(index=False))

Saved RDP analysis → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\metrics_csv\rdp_composition_analysis.csv

=== Composed (ε, δ=1e-5) vs declared ε_total — T=15 ===
 eps_total  composed_eps_delta_naive  composed_eps_delta_tree  predicted_ratio_naive_tree  predicted_ratio_naive_smooth
      0.25                    0.2545                   0.2843                       1.875                         1.875
      0.50                    0.5045                   0.5343                       1.875                         1.875
      1.00                    1.0045                   1.0343                       1.875                         1.875
      2.00                    2.0045                   2.0343                       1.875                         1.875
      4.00                    4.0045                   4.0343                       1.875                         1.875
      8.00                    8.0045                   8.0343          

In [14]:
# ── Plot 1: Theoretical vs empirical noise ratio ─────────────────────────────
# Compare the ratio predicted by theory (std_naive / std_tree) to the
# empirically observed ratio (MAE_naive / MAE_smooth) from the multi-trial CSV.
# A close match validates both our mechanism implementations and our theoretical model.

results_multi = pd.read_csv(CSV_DIR / "results_cumulative_multi.csv")

# Empirical ratio for standard config (k=1, revpct=90), averaged over seeds
std_config = results_multi[
    (results_multi["max_orders_per_user_day"] == 1) &
    (results_multi["rev_clip_pct"] == 90)
]
emp_ratio = (
    std_config.groupby(["metric", "window_days", "eps_total", "mechanism"])["MAE"]
    .mean()
    .unstack("mechanism")
    .assign(emp_ratio_naive_smooth=lambda d: d["naive"] / d["smooth"])
    .reset_index()
)

# Merge with theoretical ratio from rdp_df
theory_ratio = rdp_df[["T", "eps_total", "predicted_ratio_naive_smooth"]].rename(
    columns={"T": "window_days"}
)
merged = emp_ratio.merge(theory_ratio, on=["window_days", "eps_total"], how="left")

# One subplot per metric
metrics_plot = ["DAU", "ORDERS", "REVENUE"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, metric in zip(axes, metrics_plot):
    sub = merged[merged["metric"] == metric]
    # Use one representative eps per T to keep the plot clean
    for T_val, grp in sub.groupby("window_days"):
        grp = grp.sort_values("eps_total")
        ax.scatter(grp["predicted_ratio_naive_smooth"], grp["emp_ratio_naive_smooth"],
                   label=f"T={T_val}", s=40, alpha=0.8)

    # 45-degree reference line (perfect theory-empirics agreement)
    lims = [sub[["predicted_ratio_naive_smooth", "emp_ratio_naive_smooth"]].min().min(),
            sub[["predicted_ratio_naive_smooth", "emp_ratio_naive_smooth"]].max().max()]
    ax.plot(lims, lims, "k--", lw=0.8, label="y=x (perfect agreement)")
    ax.set_title(f"{metric}\nTheoretical vs Empirical Noise Ratio (Naive/Smooth)")
    ax.set_xlabel("Predicted ratio (theory)")
    ax.set_ylabel("Observed ratio (empirical MAE)")
    ax.legend(fontsize=7, loc="upper left")

fig.tight_layout()
rdp_plot_path = PLOTS_DIR / "theory_vs_empirical_noise_ratio.png"
fig.savefig(rdp_plot_path, dpi=160)
plt.close(fig)
print("Saved →", rdp_plot_path)

# ── Plot 2: Predicted ratio vs T (shows O(T) vs O(L^1.5) divergence) ────────
fig2, ax2 = plt.subplots(figsize=(9, 5))
for eps_val in [0.25, 1.0, 4.0, 16.0]:
    sub = rdp_df[rdp_df["eps_total"] == eps_val].sort_values("T")
    ax2.plot(sub["T"], sub["predicted_ratio_naive_smooth"],
             marker="o", label=f"ε={eps_val}")
ax2.set_xlabel("Window length T (days)")
ax2.set_ylabel("Predicted noise std ratio (Naive / Smooth)")
ax2.set_title("Theoretical Naive/Smooth Noise Ratio vs T\n(grows with T — O(T) vs O(log^1.5 T))")
ax2.legend(title="ε_total")
fig2.tight_layout()
fig2.savefig(PLOTS_DIR / "theoretical_noise_ratio_vs_T.png", dpi=160)
plt.close(fig2)
print("Saved →", PLOTS_DIR / "theoretical_noise_ratio_vs_T.png")

Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\theory_vs_empirical_noise_ratio.png
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\theoretical_noise_ratio_vs_T.png


### Task 2 — Clipping Bias Analysis

Clipping introduces **systematic signed error** that is separate from random DP noise:

- When a user's true contribution exceeds the clip bound B, we truncate it → true value is
  systematically *underestimated* (downward bias on high-value days).
- DP noise is symmetric around zero, so on average it does not add directional bias.
- The combination means: **high-value days are biased downward** (clipping dominates) and
  **low-value days may be biased upward or downward** depending on whether DP noise dominates.

We measure this by computing per-day signed error (DP − truth) from the seed_id=0 parquets,
then averaging by truth-value quartile. This analysis is independent of ε (clipping bias is
deterministic), but we show it across ε values to confirm the noise-cancellation behaviour.

**Note:** ε = 8.0 and 16.0 parquets were not saved to disk (the multi-seed run only saves
seed_id=0 artefacts and these windows were not part of the original single-seed run). Analysis
covers ε ∈ {0.25, 0.5, 1.0, 2.0, 4.0}.

In [15]:
# ── Load seed_id=0 artefacts for standard config (k=1, revpct=90, win=15) ────
BIAS_TAG     = "s=2016-07-01__e=2016-07-15__win=15__k=1__revpct=90"
BOUNDED_PATH = BOUNDED_DIR / f"bounded__{BIAS_TAG}.parquet"

assert BOUNDED_PATH.exists(), f"Bounded parquet not found: {BOUNDED_PATH}"
truth_df = pd.read_parquet(BOUNDED_PATH)

# Column map: mechanism → {metric: dp_column_name}
COL_MAP = {
    "naive":  {"DAU": "DAU_DP",        "ORDERS": "ORDERS_DP",        "REVENUE": "REVENUE_DP"},
    "tree":   {"DAU": "DAU_TREE_DP",   "ORDERS": "ORDERS_TREE_DP",   "REVENUE": "REVENUE_TREE_DP"},
    "smooth": {"DAU": "DAU_SMOOTH_DP", "ORDERS": "ORDERS_SMOOTH_DP", "REVENUE": "REVENUE_SMOOTH_DP"},
}

# Only eps values for which seed_id=0 parquets were saved
EPS_BIAS = [eps for eps in EPS_TOTALS
            if (DP_DIR / f"dp__naive__eps={eps}__{BIAS_TAG}.parquet").exists()]
print(f"ε values available for bias analysis: {EPS_BIAS}")

# ── Collect per-day signed errors ─────────────────────────────────────────────
bias_rows = []
for eps in EPS_BIAS:
    for mech, col_map in COL_MAP.items():
        dp_path = DP_DIR / f"dp__{mech}__eps={eps}__{BIAS_TAG}.parquet"
        if not dp_path.exists():
            print(f"  SKIP (missing): {dp_path.name}")
            continue
        dp_df = pd.read_parquet(dp_path)

        for metric in ["DAU", "ORDERS", "REVENUE"]:
            truth_vals = truth_df[metric].astype(float)
            dp_vals    = dp_df[col_map[metric]].astype(float)
            signed_err = dp_vals - truth_vals  # positive = DP overestimates truth

            # Assign each day to a truth-value quartile (Q1=low, Q4=high)
            try:
                quartile = pd.qcut(truth_vals, q=4,
                                   labels=["Q1 (low)", "Q2", "Q3", "Q4 (high)"],
                                   duplicates="drop")
            except ValueError:
                # Fewer than 4 unique values — use rank-based bin
                quartile = pd.cut(truth_vals.rank(method="first"),
                                  bins=4, labels=["Q1 (low)", "Q2", "Q3", "Q4 (high)"])

            for day_idx in range(len(truth_vals)):
                bias_rows.append({
                    "eps_total":     eps,
                    "mechanism":     mech,
                    "metric":        metric,
                    "truth_value":   float(truth_vals.iloc[day_idx]),
                    "dp_value":      float(dp_vals.iloc[day_idx]),
                    "signed_error":  float(signed_err.iloc[day_idx]),
                    "truth_quartile": str(quartile.iloc[day_idx]),
                })

bias_df = pd.DataFrame(bias_rows)
print(f"Bias analysis rows: {len(bias_df):,}")

# ── Figure 1: Mean signed error by quartile × mechanism — eps=1.0 ─────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
EPS_SHOW = 1.0

for ax, metric in zip(axes, ["DAU", "ORDERS", "REVENUE"]):
    sub = bias_df[(bias_df["eps_total"] == EPS_SHOW) & (bias_df["metric"] == metric)]
    agg = (sub.groupby(["truth_quartile", "mechanism"])["signed_error"]
              .mean()
              .unstack("mechanism")
              .reindex(["Q1 (low)", "Q2", "Q3", "Q4 (high)"]))

    agg.plot(kind="bar", ax=ax,
             color=[MECH_COLORS[m] for m in agg.columns],
             width=0.7, edgecolor="white")
    ax.axhline(0, color="black", lw=0.9, ls="--")
    ax.set_title(f"{metric}  (ε={EPS_SHOW})\nMean Signed Error by Truth Quartile")
    ax.set_xlabel("Truth-value quartile")
    ax.set_ylabel("Mean (DP − truth)")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="Mechanism", fontsize=8)

fig.suptitle("Clipping Bias: Signed Error (DP − Truth) by Day-value Quartile\n"
             "Negative bars = underestimation (clipping dominant); "
             "Positive bars = overestimation (noise dominant)",
             y=1.02, fontsize=10)
fig.tight_layout()
bias_plot = PLOTS_DIR / "clipping_bias_quartile_eps1.png"
fig.savefig(bias_plot, dpi=160, bbox_inches="tight")
plt.close(fig)
print("Saved →", bias_plot)

# ── Figure 2: Bias by quartile across multiple ε — REVENUE only ───────────────
fig2, axes2 = plt.subplots(1, len(EPS_BIAS), figsize=(4 * len(EPS_BIAS), 4), sharey=True)
if len(EPS_BIAS) == 1:
    axes2 = [axes2]

for ax, eps in zip(axes2, EPS_BIAS):
    sub = bias_df[(bias_df["eps_total"] == eps) & (bias_df["metric"] == "REVENUE")]
    agg = (sub.groupby(["truth_quartile", "mechanism"])["signed_error"]
              .mean()
              .unstack("mechanism")
              .reindex(["Q1 (low)", "Q2", "Q3", "Q4 (high)"]))
    agg.plot(kind="bar", ax=ax,
             color=[MECH_COLORS[m] for m in agg.columns],
             width=0.7, edgecolor="white", legend=(eps == EPS_BIAS[0]))
    ax.axhline(0, color="black", lw=0.9, ls="--")
    ax.set_title(f"ε={eps}")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)

fig2.suptitle("REVENUE Clipping Bias Across ε Values\n"
              "(as ε grows, DP noise → 0; residual bias is pure clipping truncation)",
              fontsize=10)
fig2.text(0.04, 0.5, "Mean (DP − truth)", va="center", rotation="vertical")
fig2.tight_layout()
bias_plot2 = PLOTS_DIR / "clipping_bias_revenue_vs_eps.png"
fig2.savefig(bias_plot2, dpi=160, bbox_inches="tight")
plt.close(fig2)
print("Saved →", bias_plot2)

# ── Summary table: mean signed error at large ε (≈ pure clipping bias) ────────
# At ε=4.0 noise is small enough that residual bias ≈ clipping-only bias
eps_large = max(EPS_BIAS)
clipping_bias_summary = (
    bias_df[bias_df["eps_total"] == eps_large]
    .groupby(["metric", "mechanism", "truth_quartile"])["signed_error"]
    .mean()
    .unstack("truth_quartile")
)
print(f"\n=== Estimated clipping bias (ε={eps_large}, signed_error = DP − truth) ===")
print(clipping_bias_summary.to_string())

ε values available for bias analysis: [0.25, 0.5, 1, 2, 4, 8, 16]
Bias analysis rows: 945
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\clipping_bias_quartile_eps1.png
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\clipping_bias_revenue_vs_eps.png

=== Estimated clipping bias (ε=16, signed_error = DP − truth) ===
truth_quartile        Q1 (low)           Q2           Q3    Q4 (high)
metric  mechanism                                                    
DAU     naive         0.183681     0.149799     0.597591    -0.169673
        smooth       -0.128199     0.249779     0.195499     0.091551
        tree         -0.025807     0.157532    -0.091338    -0.346435
ORDERS  naive        -0.158795    -0.765067     1.662255     0.454769
        smooth        0.078573    -0.028960     0.010254    -0.032011
        tree          0.135727     0.143098    -0.147208    -0.105904
REVENUE na

### Task 3 — ε vs T Heatmaps (Safe Zone Visualization)

The safe operating zones established for T=15 need to generalise across window lengths.
These heatmaps show how each quality metric changes as both ε and T grow simultaneously,
with a contour line marking the quality threshold. The region above the contour is the
**safe zone** for that mechanism.

The key paper argument is: structured mechanisms have a much larger safe zone (the threshold
contour sits at lower ε for any given T), and their safe zone grows more slowly with T than
Naive's — which is the empirical signature of the O(log²T) vs O(T) noise scaling.

In [16]:
# ── Load results and compute mean over seeds (standard config) ────────────────
# results_multi is already loaded above; filter to standard config
std_cfg = results_multi[
    (results_multi["max_orders_per_user_day"] == 1) &
    (results_multi["rev_clip_pct"] == 90)
]

# Metrics to heatmap and their "good" thresholds
HEATMAP_METRICS = {
    "CORR_LEVEL": {"threshold": 0.90, "label": "Pearson Corr (level)", "cmap": "RdYlGn", "vmin": 0, "vmax": 1},
    "ALARM_fp_rate": {"threshold": 0.10, "label": "Alarm FP Rate",      "cmap": "RdYlGn_r","vmin": 0, "vmax": 0.5},
    "CP_f1":         {"threshold": 0.80, "label": "CP Detection F1",    "cmap": "RdYlGn",  "vmin": 0, "vmax": 1},
}

MECHANISMS = ["naive", "smooth", "tree"]
PLOT_METRICS = ["DAU", "ORDERS", "REVENUE"]  # one heatmap figure per KPI metric

for kpi in PLOT_METRICS:
    kpi_data = std_cfg[std_cfg["metric"] == kpi]

    for quality_key, qcfg in HEATMAP_METRICS.items():
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        fig.suptitle(
            f"{kpi} — {qcfg['label']} (mean over {NUM_SEEDS} seeds)\n"
            f"Dashed line = quality threshold ({qcfg['threshold']}); "
            f"above line = safe zone",
            fontsize=11,
        )

        for ax, mech in zip(axes, MECHANISMS):
            pivot = (
                kpi_data[kpi_data["mechanism"] == mech]
                .groupby(["window_days", "eps_total"])[quality_key]
                .mean()
                .unstack("eps_total")          # columns = eps values
            )
            # Rows = T (window_days), columns = eps_total
            im = ax.imshow(
                pivot.values,
                aspect="auto",
                origin="lower",
                cmap=qcfg["cmap"],
                vmin=qcfg["vmin"],
                vmax=qcfg["vmax"],
            )
            ax.set_xticks(range(len(pivot.columns)))
            ax.set_xticklabels([str(e) for e in pivot.columns], fontsize=8)
            ax.set_yticks(range(len(pivot.index)))
            ax.set_yticklabels([str(t) for t in pivot.index], fontsize=8)
            ax.set_xlabel("ε_total")
            ax.set_ylabel("T (window days)")
            ax.set_title(f"{mech.capitalize()}")

            # Draw threshold contour line
            # imshow treats data[i,j] as pixel at (j, i), so contour also uses (j,i)
            try:
                cs = ax.contour(
                    pivot.values,
                    levels=[qcfg["threshold"]],
                    colors=["black"],
                    linewidths=[1.5],
                    linestyles=["--"],
                )
                ax.clabel(cs, fmt=f"{qcfg['threshold']}", fontsize=7)
            except Exception:
                pass  # contour may fail if all values on same side of threshold

            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        fig.tight_layout()
        safe_zone_plot = PLOTS_DIR / f"heatmap_{kpi.lower()}_{quality_key.lower()}.png"
        fig.savefig(safe_zone_plot, dpi=160, bbox_inches="tight")
        plt.close(fig)
        print("Saved →", safe_zone_plot)

print("\nAll heatmaps saved.")

Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\heatmap_dau_corr_level.png
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\heatmap_dau_alarm_fp_rate.png
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\heatmap_dau_cp_f1.png
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\heatmap_orders_corr_level.png
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\heatmap_orders_alarm_fp_rate.png
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\heatmap_orders_cp_f1.png
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\heatmap_revenue_corr_level.png
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_

### Task 4 — Privacy–Utility Pareto Frontier

A single composite utility score lets us place each (mechanism, ε) configuration on a
single privacy–utility curve — the Pareto frontier.

**Composite score construction:**

Each of the four quality dimensions is normalised to [0, 1] where 1 = perfect utility:
- `u_corr`  = CORR_LEVEL                       (higher is better; already in [−1,1])
- `u_alarm` = 1 − ALARM_fp_rate                (lower FP rate = better)
- `u_cp`    = CP_f1                             (higher is better; already in [0,1])
- `u_rank`  = (KENDALL_tau + 1) / 2            (shift from [−1,1] to [0,1])

The composite is a simple equal-weight average:
  `U = 0.25 * (u_corr + u_alarm + u_cp + u_rank)`

We compute this per (mechanism, metric, eps_total) averaged across all seeds, then plot
composite utility vs ε_total. The Pareto-dominant mechanism is the one that achieves the
highest composite score at each privacy budget.

In [17]:
# ── Build composite utility score ─────────────────────────────────────────────
# Work from the full multi-trial dataframe (standard config only)

pareto_df = std_cfg.copy()

# Normalise each dimension to [0, 1] (1 = perfect utility)
pareto_df["u_corr"]  = pareto_df["CORR_LEVEL"].clip(0, 1)
pareto_df["u_alarm"] = (1 - pareto_df["ALARM_fp_rate"]).clip(0, 1)
pareto_df["u_cp"]    = pareto_df["CP_f1"].clip(0, 1)
pareto_df["u_rank"]  = ((pareto_df["KENDALL_tau"] + 1) / 2).clip(0, 1)

# Equal-weight composite — adjust weights here if needed for sensitivity check
W_CORR, W_ALARM, W_CP, W_RANK = 0.25, 0.25, 0.25, 0.25
pareto_df["composite_U"] = (
    W_CORR  * pareto_df["u_corr"]  +
    W_ALARM * pareto_df["u_alarm"] +
    W_CP    * pareto_df["u_cp"]    +
    W_RANK  * pareto_df["u_rank"]
)

# Aggregate: mean composite score over seeds, T, and clip configs
# (grouping by metric + mechanism + eps_total gives one curve per mechanism)
pareto_agg = (
    pareto_df.groupby(["metric", "mechanism", "eps_total"])["composite_U"]
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={"mean": "U_mean", "std": "U_std"})
)

# Save
pareto_csv = CSV_DIR / "pareto_composite_utility.csv"
pareto_agg.to_csv(pareto_csv, index=False)
print("Saved →", pareto_csv)

# ── Figure 1: One subplot per metric — composite U vs ε ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle(
    "Privacy–Utility Pareto Frontier (Cumulative Strategy)\n"
    "Composite U = mean(CORR_LEVEL, 1−FP_rate, CP_F1, Kendall_τ_norm); "
    "higher = better utility",
    fontsize=11,
)

for ax, metric in zip(axes, ["DAU", "ORDERS", "REVENUE"]):
    sub = pareto_agg[pareto_agg["metric"] == metric]
    for mech, grp in sub.groupby("mechanism"):
        grp = grp.sort_values("eps_total")
        color = MECH_COLORS.get(mech)
        ax.plot(grp["eps_total"], grp["U_mean"],
                marker="o", label=mech, color=color, lw=2)
        ax.fill_between(
            grp["eps_total"],
            grp["U_mean"] - grp["U_std"],
            grp["U_mean"] + grp["U_std"],
            alpha=0.15, color=color,
        )
    ax.set_xscale("log")
    ax.set_xlabel("ε_total (log scale)")
    ax.set_ylabel("Composite Utility U")
    ax.set_title(f"{metric}")
    ax.set_ylim(0, 1)
    ax.axhline(0.9, color="grey", lw=0.8, ls=":", label="U=0.9 target")
    ax.legend(title="Mechanism", fontsize=8)
    ax.grid(True, alpha=0.3)

fig.tight_layout()
pareto_plot = PLOTS_DIR / "pareto_frontier_composite_U.png"
fig.savefig(pareto_plot, dpi=160, bbox_inches="tight")
plt.close(fig)
print("Saved →", pareto_plot)

# ── Figure 2: Budget required to hit U ≥ 0.9 — bar chart comparison ──────────
target_U = 0.9
threshold_rows = []
for metric in ["DAU", "ORDERS", "REVENUE"]:
    sub = pareto_agg[pareto_agg["metric"] == metric]
    for mech, grp in sub.groupby("mechanism"):
        grp = grp.sort_values("eps_total")
        above = grp[grp["U_mean"] >= target_U]["eps_total"]
        min_eps = float(above.min()) if len(above) > 0 else float("nan")
        threshold_rows.append({"metric": metric, "mechanism": mech, "min_eps_for_U90": min_eps})

thresh_df = pd.DataFrame(threshold_rows)
print(f"\n=== Minimum ε to reach composite U ≥ {target_U} ===")
print(thresh_df.pivot(index="mechanism", columns="metric", values="min_eps_for_U90").to_string())

fig2, axes2 = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
fig2.suptitle(f"Minimum ε to reach Composite U ≥ {target_U}\n"
              "(lower is better — less privacy budget needed for same utility)",
              fontsize=11)

for ax, metric in zip(axes2, ["DAU", "ORDERS", "REVENUE"]):
    sub = thresh_df[thresh_df["metric"] == metric].copy()
    sub = sub.sort_values("min_eps_for_U90")
    colors = [MECH_COLORS.get(m, "grey") for m in sub["mechanism"]]
    bars = ax.bar(sub["mechanism"], sub["min_eps_for_U90"], color=colors, edgecolor="white")
    for bar, val in zip(bars, sub["min_eps_for_U90"]):
        label = f"ε={val:.2f}" if not pd.isna(val) else "N/A"
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                label, ha="center", va="bottom", fontsize=8)
    ax.set_title(metric)
    ax.set_xlabel("Mechanism")
    ax.set_ylabel("Min ε_total" if metric == "DAU" else "")
    ax.set_ylim(0, max(thresh_df["min_eps_for_U90"].dropna()) * 1.2 + 0.5)

fig2.tight_layout()
pareto_bar = PLOTS_DIR / "pareto_min_eps_for_U90.png"
fig2.savefig(pareto_bar, dpi=160, bbox_inches="tight")
plt.close(fig2)
print("Saved →", pareto_bar)

print("\n✓ Phase 1 analysis complete.")

Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\metrics_csv\pareto_composite_utility.csv
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\pareto_frontier_composite_U.png

=== Minimum ε to reach composite U ≥ 0.9 ===
metric      DAU  ORDERS  REVENUE
mechanism                       
naive      16.0    16.0      NaN
smooth      2.0     2.0      4.0
tree        2.0     2.0      8.0
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\_cleaned\_grid_outputs\cumulative\plots\pareto_min_eps_for_U90.png

✓ Phase 1 analysis complete.
